In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2
import torch.nn.functional as F
import random
from torch.utils.data import Dataset

training_data = datasets.CIFAR10(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

test_data = datasets.CIFAR10(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

ModuleNotFoundError: No module named 'torch.utils.data.Dataset'

In [102]:
len(train_dataloader.dataset)

50000

In [103]:
class RotationTransform:
    def __call__(self, img):
        rotations = [0, 90, 180, 270]

        angle_idx = random.randint(0, 3)
        angle = rotations[angle_idx]

        img = torchvision.transforms.functional.rotate(img, angle)
        return img, angle_idx

class RotationDataset(torch.utils.data.Dataset):
    def __init__(self):
        self.dataset = datasets.CIFAR10(
            root="data",
            train=True,
            download=True,
            transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
        )
        self.rotation_transform = RotationTransform()

    def __len__(self):
        return (len(self.dataset))
    
    def __getitem__(self, idx):
        img, _ = self.dataset[idx]
        img, rot_label = self.rotation_transform(img)
        return img, rot_label

In [104]:
dataset = RotationDataset()
img, label = dataset[0]
print(img.shape)
print(label)

NameError: name 'torchvision' is not defined

In [74]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10, bias=False)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        logits = self.fc3(x)
        return logits
    
model = Net()

In [73]:
learning_rate = 4e-3
batch_size = 64
epochs = 5

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using mps device


In [77]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.to(device)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X = X.to(device)
        y = y.to(device)
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, curr = loss.item(), batch*batch_size + len(X)
            print(f"loss: {loss:>7f} [{curr:>5d}/{size:>5d}]")

def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct)::>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [81]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

epochs = 50
for t in range(epochs):
    print(f"Epoch {t+1}\n-----------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-----------------------------------
loss: 1.017395 [   64/50000]
loss: 1.069044 [ 6464/50000]
loss: 0.809798 [12864/50000]
loss: 0.904897 [19264/50000]
loss: 1.140394 [25664/50000]
loss: 1.087141 [32064/50000]
loss: 0.904489 [38464/50000]
loss: 0.849485 [44864/50000]
Test Error: 
 Accuracy: 53.3%, Avg loss: 1.544897 

Epoch 2
-----------------------------------
loss: 0.892467 [   64/50000]
loss: 1.167837 [ 6464/50000]
loss: 0.716560 [12864/50000]
loss: 0.860960 [19264/50000]
loss: 1.099900 [25664/50000]
loss: 1.009553 [32064/50000]
loss: 1.008900 [38464/50000]
loss: 0.840137 [44864/50000]
Test Error: 
 Accuracy: 53.7%, Avg loss: 1.546327 

Epoch 3
-----------------------------------
loss: 0.936254 [   64/50000]
loss: 1.316523 [ 6464/50000]
loss: 0.678608 [12864/50000]
loss: 0.801279 [19264/50000]
loss: 1.033084 [25664/50000]
loss: 0.894766 [32064/50000]
loss: 0.930094 [38464/50000]
loss: 0.833740 [44864/50000]
Test Error: 
 Accuracy: 52.6%, Avg loss: 1.589536 

Epoch 4
--------